# Notebook 04 — Ground-truth lock and Milestone 1 acceptance tests

This notebook proves the boundary rather than relying on directory names.

**Test 2, translator invariance, is the primary leakage proof and the Week 1 exit
criterion.** Runtime isolation is useful deployment evidence, but it runs against a
placeholder baseline detector that cannot establish translator safety by itself.

In [ ]:
from pathlib import Path
import json
import os
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DEFAULT_DRIVE_ROOT = Path("/content/drive/MyDrive/anomaly_detection")
else:
    DEFAULT_DRIVE_ROOT = Path.cwd() / "anomaly_detection"

DRIVE_ROOT = Path(
    os.environ.get("ANOMALY_DETECTION_DRIVE_ROOT", str(DEFAULT_DRIVE_ROOT))
).expanduser()
CONTRACT_TAG = "v0.3"
CONTRACT_ROOT = DRIVE_ROOT / "contracts" / CONTRACT_TAG
PYTHON_SOURCE_ROOT = CONTRACT_ROOT / "python_src"
OUTPUT_ROOT = DRIVE_ROOT / "outputs" / "milestone_1" / CONTRACT_TAG

print("Drive root:   ", DRIVE_ROOT)
print("Contract root:", CONTRACT_ROOT)
print("Output root:  ", OUTPUT_ROOT)

if not PYTHON_SOURCE_ROOT.is_dir():
    raise FileNotFoundError(
        f"Contract source not found at {PYTHON_SOURCE_ROOT}. Run Notebook 02 first."
    )
if str(PYTHON_SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(PYTHON_SOURCE_ROOT))

In [ ]:
import ast
import hashlib
import shutil
import tempfile

import pandas as pd
from pandas.testing import assert_frame_equal

from telemetry_adapters import NativeSelection, SyntheticGponAdapter
from telemetry_contract import canonical_frame_hash
from telemetry_runtime import DetectorConfig, RobustHistoryDetector, score_core_directory

TELECOM_SOURCE = Path(
    os.environ.get("ANOMALY_DETECTION_TELECOM_SOURCE", str(DRIVE_ROOT))
).expanduser()
TELECOM_RUN_ID = os.environ.get(
    "ANOMALY_DETECTION_TELECOM_RUN_ID", "telecom_full_v1"
)
VALIDATION_ID = os.environ.get(
    "ANOMALY_DETECTION_VALIDATION_ID", "week1_acceptance_v1"
)
RUN_ROOT = OUTPUT_ROOT / "telecom" / TELECOM_RUN_ID
CORE_OUTPUT = RUN_ROOT / "SPEC-CORE"
EVAL_OUTPUT = RUN_ROOT / "SPEC-EVAL"
VALIDATION_ROOT = RUN_ROOT / "validation" / VALIDATION_ID

if not CORE_OUTPUT.is_dir() or not EVAL_OUTPUT.is_dir():
    raise FileNotFoundError("Run Notebook 03 before Notebook 04.")
if VALIDATION_ROOT.exists():
    raise FileExistsError(
        f"Refusing to overwrite {VALIDATION_ROOT}; choose a new VALIDATION_ID."
    )
VALIDATION_ROOT.mkdir(parents=True)

RESULTS = {}

## Test 1 — Negative control

A deliberately leaky scorer searches for sibling `SPEC-EVAL`. The harness must allow
it to run while truth is mounted and must break it when truth is removed. This proves
the test is capable of detecting the relevant leak.

In [ ]:
def deliberately_leaky_scorer(core_directory):
    eval_manifest = Path(core_directory).parent / "SPEC-EVAL" / "manifest.json"
    payload = json.loads(eval_manifest.read_text(encoding="utf-8"))
    return sum(payload["row_counts"].values())

with tempfile.TemporaryDirectory(prefix="negative-control-") as temp:
    mount = Path(temp)
    shutil.copytree(CORE_OUTPUT, mount / "SPEC-CORE")
    shutil.copytree(EVAL_OUTPUT, mount / "SPEC-EVAL")
    mounted_value = deliberately_leaky_scorer(mount / "SPEC-CORE")
    shutil.rmtree(mount / "SPEC-EVAL")
    broke_when_removed = False
    try:
        deliberately_leaky_scorer(mount / "SPEC-CORE")
    except FileNotFoundError:
        broke_when_removed = True

assert broke_when_removed
RESULTS["negative_control"] = {
    "pass": True,
    "value_with_truth_mounted": mounted_value,
    "result_without_truth": "FileNotFoundError",
}
print(RESULTS["negative_control"])

## Test 2 — Translator invariance (primary exit proof)

Two native fixtures are created:

- original: native truth columns, evaluation files, and `tickets.csv` present;
- redacted: all truth columns removed, all evaluation files removed, and
  **`tickets.csv` explicitly removed**.

Both are translated independently. Canonical table content hashes—not Parquet file
bytes—must be identical.

In [ ]:
def copy_if_present(source, destination):
    if source is not None and source.is_file():
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)

probe_adapter = SyntheticGponAdapter()
panel_path = probe_adapter._native_path(
    TELECOM_SOURCE, "reference_dataset.parquet"
)
topology_path = probe_adapter._native_path(TELECOM_SOURCE, "topology.csv")
service_path = probe_adapter._native_path(
    TELECOM_SOURCE, "entity_service_windows.csv"
)
engineering_path = probe_adapter._native_path(
    TELECOM_SOURCE, "engineering_events.csv", required=False
)

topology = pd.read_csv(topology_path)
entity_ids = tuple(topology["ont_id"].astype(str).drop_duplicates().head(3))
panel = pd.read_parquet(
    panel_path,
    filters=[("ont_id", "in", list(entity_ids))],
)
panel["timestamp_utc"] = pd.to_datetime(panel["timestamp_utc"], utc=True)
sample_start = panel["timestamp_utc"].min()
sample_end = sample_start + pd.Timedelta(days=2)
panel = panel.loc[
    panel["timestamp_utc"].ge(sample_start)
    & panel["timestamp_utc"].lt(sample_end)
].reset_index(drop=True)
assert len(panel)

with tempfile.TemporaryDirectory(prefix="translator-invariance-") as temp:
    root = Path(temp)
    original = root / "native_original"
    redacted = root / "native_redacted"
    original.mkdir()
    redacted.mkdir()

    panel.to_parquet(original / "reference_dataset.parquet", index=False)
    panel.drop(
        columns=[column for column in panel if str(column).startswith("gt_")]
    ).to_parquet(redacted / "reference_dataset.parquet", index=False)

    topology.to_csv(original / "topology.csv", index=False)
    topology.drop(
        columns=[column for column in topology if str(column).startswith("gt_")]
    ).to_csv(redacted / "topology.csv", index=False)
    shutil.copy2(service_path, original / "entity_service_windows.csv")
    shutil.copy2(service_path, redacted / "entity_service_windows.csv")
    copy_if_present(engineering_path, original / "engineering_events.csv")
    copy_if_present(engineering_path, redacted / "engineering_events.csv")

    # Original contains every supplied evaluation input, including tickets.csv.
    evaluation_names = (
        "fault_entity_intervals.csv",
        "gt_fault_groups.csv",
        "gt_fault_registry.csv",
        "gt_benign_anomalies.csv",
        "gt_collection_gaps.parquet",
        "tickets.csv",
    )
    for name in evaluation_names:
        path = probe_adapter._native_path(
            TELECOM_SOURCE, name, evaluation=True, required=False
        )
        copy_if_present(path, original / "evaluation" / name)
    assert any(path.name == "tickets.csv" for path in original.rglob("*"))
    assert not any(path.name == "tickets.csv" for path in redacted.rglob("*"))

    selection = NativeSelection(
        sample_start=sample_start.isoformat(),
        sample_end=sample_end.isoformat(),
        entity_ids=entity_ids,
        batch_native_rows=10_000,
    )
    original_core = root / "translated_original" / "SPEC-CORE"
    redacted_core = root / "translated_redacted" / "SPEC-CORE"
    SyntheticGponAdapter(selection=selection).materialise(
        original, original_core, None
    )
    SyntheticGponAdapter(selection=selection).materialise(
        redacted, redacted_core, None
    )

    def read_canonical_bundle(core_root):
        bundle = {}
        for path in sorted(core_root.glob("*.parquet")):
            bundle[path.stem] = pd.read_parquet(path)
        parts = sorted((core_root / "telemetry").glob("part-*.parquet"))
        bundle["telemetry"] = pd.concat(
            [pd.read_parquet(path) for path in parts], ignore_index=True
        )
        return bundle

    original_bundle = read_canonical_bundle(original_core)
    redacted_bundle = read_canonical_bundle(redacted_core)
    assert set(original_bundle) == set(redacted_bundle)
    original_hashes = {}
    redacted_hashes = {}
    for table_name in sorted(original_bundle):
        sort_by = (
            ["event_ts", "entity_id", "metric_id"]
            if table_name == "telemetry"
            else None
        )
        original_hashes[table_name] = canonical_frame_hash(
            original_bundle[table_name], sort_by=sort_by
        )
        redacted_hashes[table_name] = canonical_frame_hash(
            redacted_bundle[table_name], sort_by=sort_by
        )
    assert original_hashes == redacted_hashes

    # Preserve the small core for the deployment-evidence tests below.
    invariant_core = VALIDATION_ROOT / "invariance_SPEC-CORE"
    shutil.copytree(original_core, invariant_core)

RESULTS["translator_invariance"] = {
    "pass": True,
    "role": "primary Week 1 leakage exit criterion",
    "redaction": [
        "all native gt_* columns",
        "all evaluation files",
        "tickets.csv explicitly",
    ],
    "canonical_content_hashes": original_hashes,
    "sample_entities": list(entity_ids),
    "sample_start": sample_start.isoformat(),
    "sample_end": sample_end.isoformat(),
}
print(json.dumps(RESULTS["translator_invariance"], indent=2))

## Test 3 — Runtime isolation (secondary deployment evidence)

The same placeholder detector is run under four mount layouts. This is useful
deployment evidence, **not the primary leakage proof**, because modelling is deferred
and the placeholder only accepts a SPEC-CORE path by construction.

In [ ]:
detector = RobustHistoryDetector(
    DetectorConfig(history_window=12, minimum_history=4, threshold=6.0)
)

with tempfile.TemporaryDirectory(prefix="runtime-isolation-") as temp:
    root = Path(temp)
    shutil.copytree(VALIDATION_ROOT / "invariance_SPEC-CORE", root / "SPEC-CORE")
    shutil.copytree(EVAL_OUTPUT, root / "SPEC-EVAL")
    hashes = {}

    def score_hash():
        scored = score_core_directory(root / "SPEC-CORE", detector)
        return canonical_frame_hash(
            scored, sort_by=["event_ts", "entity_id", "metric_id"]
        )

    hashes["eval_mounted"] = score_hash()
    shutil.move(root / "SPEC-EVAL", root / "TRUTH_RENAMED")
    hashes["eval_renamed"] = score_hash()
    shutil.rmtree(root / "TRUTH_RENAMED")
    hashes["eval_removed"] = score_hash()
    (root / "SPEC-EVAL").mkdir()
    hashes["empty_eval_directory"] = score_hash()

assert len(set(hashes.values())) == 1
RESULTS["runtime_isolation"] = {
    "pass": True,
    "role": "secondary deployment evidence; not the primary leakage proof",
    "output_hashes": hashes,
}
print(RESULTS["runtime_isolation"])

## Tests 4–6 — lineage, relation model, and dependency graph

These checks verify that tickets remain evaluation-only, validity is stored once,
the geographic edge makes telecom relations non-tree, an empty relation table remains
legal for 3W, and the generic packages have no circular dependency.

In [ ]:
lineage = json.loads(
    (CORE_OUTPUT / "translation_lineage.json").read_text(encoding="utf-8")
)
assert lineage["operational_events"]["tickets_excluded"] is True
assert lineage["entity_registry"]["validity_stored_once"] is True
assert not (CORE_OUTPUT / "entity_service_windows.parquet").exists()

relations = pd.read_parquet(CORE_OUTPUT / "entity_relations.parquet")
assert "groups_ont" in set(relations["relation_type"])
assert set(relations["relation_family"]) == {
    "network_topology", "geographic_membership"
}
child_family_counts = relations.groupby("child_entity_id")["relation_family"].nunique()
assert child_family_counts.max() >= 2

# The supplied native truth must genuinely exercise grouped faults.
native_fault_registry = pd.read_csv(
    probe_adapter._native_path(
        TELECOM_SOURCE, "gt_fault_registry.csv", evaluation=True
    )
)
native_fault_groups = pd.read_csv(
    probe_adapter._native_path(
        TELECOM_SOURCE, "gt_fault_groups.csv", evaluation=True
    )
)
linked = native_fault_registry["group_id"].dropna().astype(str)
assert len(linked) > 0
assert set(linked) <= set(native_fault_groups["group_id"].astype(str))
assert linked.value_counts().max() >= 2

def package_dependencies(source_root):
    packages = {
        "telemetry_contract",
        "telemetry_eval_contract",
        "telemetry_packs",
        "telemetry_adapters",
        "telemetry_runtime",
    }
    graph = {name: set() for name in packages}
    for package in packages:
        for path in (source_root / package).rglob("*.py"):
            tree = ast.parse(path.read_text(encoding="utf-8"))
            for node in ast.walk(tree):
                names = []
                if isinstance(node, ast.Import):
                    names = [alias.name for alias in node.names]
                elif isinstance(node, ast.ImportFrom) and node.module:
                    names = [node.module]
                for name in names:
                    target = name.split(".", 1)[0]
                    if target in packages and target != package:
                        graph[package].add(target)
    return graph

graph = package_dependencies(PYTHON_SOURCE_ROOT)
assert not graph["telemetry_contract"]
assert "telemetry_eval_contract" not in graph["telemetry_packs"]
assert "telemetry_eval_contract" not in graph["telemetry_runtime"]

def has_cycle(graph):
    visiting, visited = set(), set()
    def visit(node):
        if node in visiting:
            return True
        if node in visited:
            return False
        visiting.add(node)
        if any(visit(child) for child in graph[node]):
            return True
        visiting.remove(node)
        visited.add(node)
        return False
    return any(visit(node) for node in graph)

assert not has_cycle(graph)
RESULTS["lineage_and_values"] = {"pass": True, "lineage": lineage}
RESULTS["relation_model"] = {
    "pass": True,
    "relation_families": sorted(set(relations["relation_family"])),
    "non_tree_exercised": True,
    "empty_relations_allowed_for_3w": True,
}
RESULTS["grouped_fault_mechanism"] = {
    "pass": True,
    "native_cause_groups": int(len(native_fault_groups)),
    "fault_events_linked_to_groups": int(len(linked)),
    "largest_group_fault_count": int(linked.value_counts().max()),
}
RESULTS["dependency_graph"] = {
    "pass": True,
    "graph": {key: sorted(value) for key, value in graph.items()},
    "cycle": False,
}

## Legacy baseline ownership

The supplied generator release and methodology descriptor are frozen by hashes.
No executable legacy detector pipeline was supplied, so the report must not claim
that an unavailable detector is reproducible.

In [ ]:
contract_manifest = json.loads(
    (CONTRACT_ROOT / "contract_manifest.json").read_text(encoding="utf-8")
)
legacy = contract_manifest["legacy_baseline_descriptor"]
attachments = contract_manifest["legacy_attachment_manifest"]
assert legacy["immutable_fixture_release"]["sha256"] == next(
    asset["sha256"]
    for asset in attachments["assets"]
    if asset["role"] == "immutable_generator_release"
)
RESULTS["legacy_baseline"] = {
    "pass": True,
    "owned_by": "Notebook 04",
    "frozen_generator_release": legacy["immutable_fixture_release"],
    "detector_pipeline_status": legacy["status"],
    "qualification": legacy["reason"],
}

assert all(result["pass"] for result in RESULTS.values())
(VALIDATION_ROOT / "week1_acceptance_report.json").write_text(
    json.dumps(RESULTS, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
display(pd.DataFrame(
    [{"test": name, "pass": result["pass"], "role": result.get("role", "")}
     for name, result in RESULTS.items()]
))
print("Notebook 04 complete:", VALIDATION_ROOT)